# Lecture 7 — Class Exercise
## Heatmap & Waterfall: Netflix Catalogue

> **Push to:** `week07/lecture07_exercise.ipynb`

**Rules:**
1. Heatmap: colour scale must match the data type (sequential for counts, diverging for above/below)
2. Waterfall: use green for additions, red for subtractions, blue for totals
3. Insight title tells the setup-conflict-resolution story (or at minimum states the finding)
4. Annotate at least one cell or bar directly

---


In [2]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv(r"C:\Users\Sakshit\Desktop\clg projects\SEM 2\Data Visualization\data-viz-class-material\data\netflix_catalogue.csv")
print(f"Loaded: {len(df)} titles")
print(df['type'].value_counts())
print(df.head())


Loaded: 3000 titles
type
Movie      1974
TV Show    1026
Name: count, dtype: int64
      type  release_year  added_year             genre        country rating  \
0    Movie          2014        2016  Sci-Fi & Fantasy         France  PG-13   
1    Movie          2010        2014     Documentaries  United States  TV-MA   
2  TV Show          2011        2012     Kids & Family  United States  TV-14   
3    Movie          2016        2018             Anime          India     PG   
4    Movie          2014        2016     Kids & Family         Canada  TV-MA   

   duration  
0       157  
1       127  
2         6  
3       134  
4        77  


In [2]:
print("Genres:", df['genre'].value_counts().head(8))
print("\nCountries:", df['country'].value_counts().head(8))
print("\nRatings:", df['rating'].value_counts())


Genres: genre
Sports                244
Sci-Fi & Fantasy      213
Kids & Family         209
Crime                 206
Drama                 204
Horror                199
Action & Adventure    198
Thrillers             195
Name: count, dtype: int64

Countries: country
United States     932
India             337
United Kingdom    261
Japan             187
France            176
Canada            164
South Korea       151
Mexico            138
Name: count, dtype: int64

Ratings: rating
TV-MA    840
TV-14    733
PG-13    589
R        312
PG       196
TV-PG    128
G         92
TV-Y7     57
TV-G      53
Name: count, dtype: int64


## Task 1 — Heatmap: content by rating and release decade

**What to build:** A heatmap showing the number of titles by **content rating** (y-axis) and **decade** (x-axis).

**Requirements:**
- Create a 'decade' column: `df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'`
- Filter to TV-14, TV-MA, PG-13, R, PG only (most common ratings)
- Sequential colour scale (Blues)
- Values shown in cells (`text_auto=True`)
- Insight title about which rating dominates which decade


In [4]:
# Task 1
# YOUR CODE HERE
import pandas as pd
import plotly.express as px

# Load dataset
df = pd.read_csv(r"C:\Users\Sakshit\Desktop\clg projects\SEM 2\Data Visualization\data-viz-class-material\data\netflix_catalogue.csv")

# Create decade column
df["decade"] = (df["release_year"] // 10 * 10).astype(str) + "s"

# Filter to common ratings
common_ratings = ["TV-14", "TV-MA", "PG-13", "R", "PG"]
filtered_df = df[df["rating"].isin(common_ratings)]

# Count titles by rating and decade
heatmap_data = (
    filtered_df.groupby(["rating", "decade"])
    .size()
    .reset_index(name="count")
)

# Plot heatmap
fig = px.imshow(
    heatmap_data.pivot(index="rating", columns="decade", values="count"),
    color_continuous_scale="Blues",
    text_auto=True,
    labels=dict(x="Decade", y="Rating", color="Number of Titles"),
    title="Heatmap of Netflix Titles by Rating and Release Decade"
)
fig.show()


## Task 2 — Waterfall: Movie vs TV Show additions by year

**What to build:** A waterfall chart showing how Netflix's **Movie library** grew year by year (2015-2022).

**Requirements:**
- Filter to Movies only
- Group by `added_year`, count titles per year
- Final bar should be the cumulative total
- Green bars (additions), blue total
- Annotation on the year with the largest single addition
- Insight title naming the growth story


In [5]:
# Task 2
# YOUR CODE HERE
import pandas as pd
import plotly.graph_objects as go

# Load dataset
df = pd.read_csv(r"C:\Users\Sakshit\Desktop\clg projects\SEM 2\Data Visualization\data-viz-class-material\data\netflix_catalogue.csv")

# Filter to Movies only
movies_df = df[df["type"] == "Movie"]

# Group by added_year and count titles
yearly_additions = (
    movies_df.groupby("added_year")
    .size()
    .reset_index(name="additions")
    .sort_values("added_year")
)

# Compute cumulative total
yearly_additions["cumulative_total"] = yearly_additions["additions"].cumsum()

# Identify year with largest single addition
max_year = yearly_additions.loc[yearly_additions["additions"].idxmax(), "added_year"]
max_value = yearly_additions["additions"].max()

# Create waterfall chart
fig = go.Figure()

# Add bars for yearly additions (green)
fig.add_trace(go.Waterfall(
    name="Additions",
    orientation="v",
    measure=["relative"] * len(yearly_additions) + ["total"],
    x=list(yearly_additions["added_year"]) + ["Total"],
    y=list(yearly_additions["additions"]) + [yearly_additions["cumulative_total"].iloc[-1]],
    textposition="outside",
    connector={"line": {"color": "rgba(63, 63, 63, 0.5)"}},
    increasing={"marker": {"color": "green"}},
    totals={"marker": {"color": "blue"}}
))

# Annotate the year with largest addition
fig.add_annotation(
    x=max_year,
    y=max_value,
    text=f"Largest addition: {max_year} ({max_value} titles)",
    showarrow=True,
    arrowhead=2,
    ax=0,
    ay=-40
)

# Add title and labels
fig.update_layout(
    title="Netflix Movie Library Growth (2015–2022)",
    xaxis_title="Year Added",
    yaxis_title="Number of Movies Added",
    waterfallgap=0.3
)

fig.show()
